# Code Generator (Trình sinh mã)

Yêu cầu: dùng Frontier model (mô hình hàng đầu) để sinh mã C++ hiệu năng cao từ mã Python


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Nhắc nhở: TÙY CHỌN khi thực thi mã C++ hoặc Rust</h2>
            <span style="color:#f71;">Cách khác: bạn có thể chạy trên website đã giới thiệu hôm qua</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Lưu ý quan trọng</h1>
            <span style="color:#900;">
            Trong lab (bài thực hành) này, mình dùng các model cao cấp GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4 — những model có giá hơi cao hơn. Chi phí vẫn thấp, nhưng nếu bạn muốn giữ chi phí cực thấp, hãy chọn model rẻ hơn như gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# imports (các thư viện)

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("Chưa thiết lập OpenAI API Key")
    
if anthropic_api_key:
    print(f"Anthropic API Key tồn tại và bắt đầu bằng {anthropic_api_key[:7]}")
else:
    print("Chưa thiết lập Anthropic API Key (và đây là tùy chọn)")

if google_api_key:
    print(f"Google API Key tồn tại và bắt đầu bằng {google_api_key[:2]}")
else:
    print("Chưa thiết lập Google API Key (và đây là tùy chọn)")

if grok_api_key:
    print(f"Grok API Key tồn tại và bắt đầu bằng {grok_api_key[:4]}")
else:
    print("Chưa thiết lập Grok API Key (và đây là tùy chọn)")

if groq_api_key:
    print(f"Groq API Key tồn tại và bắt đầu bằng {groq_api_key[:4]}")
else:
    print("Chưa thiết lập Groq API Key (và đây là tùy chọn)")

if openrouter_api_key:
    print(f"OpenRouter API Key tồn tại và bắt đầu bằng {openrouter_api_key[:6]}")
else:
    print("Chưa thiết lập OpenRouter API Key (và đây là tùy chọn)")



In [ ]:
# Kết nối các client library (thư viện client)

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [ ]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "gemini-2.5-pro": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Muốn giữ chi phí cực thấp? Thay bằng các model bạn chọn, dùng ví dụ từ hôm qua

In [ ]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

In [ ]:
message = f"""
Đây là báo cáo thông tin hệ thống (system information) của máy tính tôi.
Tôi muốn chạy Rust compiler (trình biên dịch Rust) để biên dịch một file rust tên main.rs rồi thực thi theo cách đơn giản nhất.
Hãy trả lời xem tôi có cần cài Rust toolchain (bộ công cụ Rust) không. Nếu có, hãy đưa hướng dẫn từng bước đơn giản nhất.

Nếu máy tôi đã sẵn sàng biên dịch Rust, tôi muốn chạy đoạn Python tương tự như sau để biên dịch và thực thi:
```python
compile_command = # điền lệnh ở đây — để đạt runtime performance (hiệu năng khi chạy) nhanh nhất có thể
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # điền lệnh ở đây
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Hãy cho tôi chính xác nên dùng gì cho compile_command và run_command.
Ưu tiên runtime performance tối đa; compile time (thời gian biên dịch) có thể chậm. Điều quan trọng là chạy nhanh nhất trên platform (nền tảng) này.
Trả lời các lệnh bằng markdown.

Thông tin hệ thống:
{system_info}

Thông tin Rust toolchain:
{rust_info}
"""

response = openai.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

## Với C++, ghi đè bằng lệnh từ hôm qua; với Rust, dùng các lệnh mới

Hoặc dùng website như hôm qua:

 https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
compile_command = [
    "/Users/ed/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "-o", "main",
]

run_command = ["./main"]


## Tiếp theo: nhiệm vụ chính

In [ ]:
language = "Rust" # hoặc "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Nhiệm vụ của bạn là chuyển mã Python thành mã {language} hiệu năng cao (high performance).
Chỉ trả lời bằng mã {language}. Không giải thích, trừ một vài comment (chú thích) khi cần.
Mã {language} phải cho ra output (kết quả in ra) giống hệt, trong thời gian ngắn nhất có thể.
"""

def user_prompt_for(python):
    return f"""
Port (chuyển) mã Python này sang {language} với implementation (cách hiện thực) nhanh nhất, cho ra output giống hệt trong thời gian ngắn nhất.
Thông tin hệ thống là:
{system_info}
Phản hồi của bạn sẽ được ghi vào file tên main.{language} rồi biên dịch và thực thi; lệnh compilation (biên dịch) là:
{compile_command}
Chỉ trả lời bằng mã {language}.
Mã Python cần port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [ ]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [ ]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Lỗi (Error): {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
# Dùng các lệnh từ GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"Đã xảy ra lỗi (error):\n{e.stderr}"

In [ ]:
python_hard = """# Cẩn thận hỗ trợ số lớn (large numbers)

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Tham số (Parameters)
n = 10000         # Số lượng số ngẫu nhiên (Number of random numbers)
initial_seed = 42 # Seed ban đầu cho LCG (Initial seed)
min_val = -10     # Giá trị nhỏ nhất của số ngẫu nhiên (Minimum value)
max_val = 10      # Giá trị lớn nhất của số ngẫu nhiên (Maximum value)

# Đo thời gian hàm (Timing the function)
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Tổng Maximum Subarray Sum (20 lần chạy):", result)
print("Thời gian thực thi (Execution Time): {:.6f} giây".format(end_time - start_time))
"""

In [ ]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port (chuyển) từ Python sang {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (bản gốc)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (do model sinh)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Chạy Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port sang {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Chạy {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Kết quả Python", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"Kết quả {language}", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


## KẾT QUẢ!

Qwen 2.5 Coder: FAIL (thất bại)  
Gemini 2.5 Pro: FAIL (thất bại)  
DeepSeek Coder v2: FAIL (thất bại)  
Qwen3 Coder 30B: FAIL (thất bại)  
Claude Sonnet 4.5: FAIL (thất bại)    
GPT-5: FAIL (thất bại)    

Hạng 3: GPT-oss-20B: 0.000341  
Hạng 2: Grok 4: 0.000317  
**Hạng 1: OpenAI GPT-OSS 120B: 0.000304**  

In [ ]:
print(f"Trong thí nghiệm của Ed, kết quả model GPT-OSS 120B nhanh hơn mã Python {33.755209/0.000304:,.0f} lần.")